# 🐄 E-Gowshala — Automated Cattle Disease CNN Training
### End-to-End Deep Learning Pipeline (EfficientNetB0)

This notebook will:
1. ✅ Authenticate with Kaggle using preconfigured credentials
2. 📥 Download all verified cattle disease datasets (LSD, FMD, Mastitis, Lameness, Skin, Healthy)
3. 🗂️ Merge & structure images into unified train/val/test splits
4. 🧠 Train an EfficientNetB0 Transfer Learning model with Data Augmentation
5. 📊 Evaluate with Confusion Matrix & Accuracy curves
6. 📦 Export `cattle_disease_v1.h5` and `cattle_disease_v1.tflite` for the E-Gowshala app

In [ ]:
# ── STEP 1: Verify GPU & Install Dependencies ──────────────────
import subprocess, sys, os, json, shutil
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Ready: {gpus[0].name}")
else:
    print("⚠️ No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU -> Save")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle", "Pillow", "matplotlib", "seaborn", "scikit-learn"])
print("✅ Packages installed successfully!")

In [ ]:
# ── STEP 2: Configure Kaggle Authentication ────────────────────
KAGGLE_USERNAME = "vanshikayadav084"
KAGGLE_KEY      = "KGAT_3c9707a94d7b1a898ba71c6f74ed41e1"

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("✅ Kaggle credentials configured!")

In [ ]:
# ── STEP 3: Download Verified Datasets ─────────────────────────
print("📥 Downloading image datasets from Kaggle...")

datasets = [
    ("warcoder/lumpy-skin-images-dataset",                  "/content/dl/lsd_warcoder"),
    ("shivamagarwal29/cow-lumpy-disease-dataset",           "/content/dl/lsd_shivam"),
    ("wasimfaraz/fmd-cattle-dataset",                       "/content/dl/fmd_wasim"),
    ("wasimfaraz/cattle-foot-and-mouth-disease-fmd",         "/content/dl/fmd_extended"),
    ("devang03mgr/cattle-diseases-datasets",                  "/content/dl/cattle_multi"),
    ("magantirajasri/cattle-diseases-dataset",                "/content/dl/cattle_maganti"),
]

for slug, target_dir in datasets:
    os.makedirs(target_dir, exist_ok=True)
    print(f"  -> Downloading {slug}...")
    os.system(f"kaggle datasets download -d {slug} -p {target_dir} --unzip -q")
    print(f"     Done: {slug}")

print("\n✅ All datasets downloaded and extracted!")

In [ ]:
# ── STEP 4: Structure Data into Classes ────────────────────────
from pathlib import Path
from PIL import Image
import random

DATASET_ROOT = Path("/content/dataset")
CLASSES = [
    "healthy",
    "lumpy_skin_disease",
    "foot_mouth_disease",
    "mastitis",
    "lameness",
    "eye_disease",
    "skin_disease",
    "respiratory_disease"
]

for split in ["train", "val", "test"]:
    for cls in CLASSES:
        (DATASET_ROOT / split / cls).mkdir(parents=True, exist_ok=True)

def collect_and_distribute(source_dirs, cls_name, max_count=800):
    imgs = []
    for s in source_dirs:
        p = Path(s)
        if p.exists():
            for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"]:
                imgs.extend(list(p.rglob(ext)))
    
    # Fallback keyword scan if few images found
    if len(imgs) < 30:
        for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG"]:
            for found in Path("/content/dl").rglob(ext):
                if cls_name.replace("_", " ") in str(found).lower() or cls_name in str(found).lower():
                    if found not in imgs:
                        imgs.append(found)

    imgs = list(set(imgs))
    random.seed(42)
    random.shuffle(imgs)
    imgs = imgs[:max_count]

    n_train = int(len(imgs) * 0.70)
    n_val   = int(len(imgs) * 0.20)

    splits = {
        "train": imgs[:n_train],
        "val":   imgs[n_train:n_train+n_val],
        "test":  imgs[n_train+n_val:]
    }

    counts = {}
    for split, split_imgs in splits.items():
        dest_folder = DATASET_ROOT / split / cls_name
        valid = 0
        for i, img_path in enumerate(split_imgs):
            try:
                with Image.open(img_path) as im:
                    im.verify()
                shutil.copy2(img_path, dest_folder / f"{cls_name}_{i:05d}{img_path.suffix.lower()}")
                valid += 1
            except:
                pass
        counts[split] = valid
    return counts

print("Organizing classes into dataset folder...")
class_sources = {
    "lumpy_skin_disease": ["/content/dl/lsd_warcoder/Lumpy Skin", "/content/dl/lsd_shivam/Lumpy", "/content/dl/cattle_multi/lumpy skin"],
    "healthy":            ["/content/dl/lsd_warcoder/Normal Skin", "/content/dl/lsd_shivam/Normal", "/content/dl/cattle_multi/Normal"],
    "foot_mouth_disease": ["/content/dl/fmd_wasim", "/content/dl/fmd_extended", "/content/dl/cattle_multi/Foot and Mouth"],
    "mastitis":           ["/content/dl/cattle_multi/mastitis", "/content/dl/cattle_maganti/Udder"],
    "lameness":           ["/content/dl/cattle_multi/lameness", "/content/dl/cattle_maganti/Foot"],
    "eye_disease":        ["/content/dl/cattle_multi/eye", "/content/dl/cattle_maganti/eye"],
    "skin_disease":       ["/content/dl/cattle_multi/ringworm", "/content/dl/cattle_multi/dermatitis"],
    "respiratory_disease":["/content/dl/cattle_multi/respiratory", "/content/dl/cattle_multi/nasal"],
}

print(f"{'Class':<25} {'Train':>6} {'Val':>6} {'Test':>6}")
print("-" * 50)
for cls, sources in class_sources.items():
    c = collect_and_distribute(sources, cls)
    print(f"{cls:<25} {c.get('train',0):>6} {c.get('val',0):>6} {c.get('test',0):>6}")
print("\n✅ Dataset organized successfully!")

In [ ]:
# ── STEP 5: Create Data Generators with Augmentation ───────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode="nearest"
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    DATASET_ROOT / "train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    DATASET_ROOT / "val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_gen = val_datagen.flow_from_directory(
    DATASET_ROOT / "test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

CLASS_NAMES = {str(v): k for k, v in train_gen.class_indices.items()}
with open("/content/class_mapping.json", "w") as f:
    json.dump({"index_to_class": CLASS_NAMES, "class_to_index": train_gen.class_indices}, f, indent=2)

print(f"\n✅ Found {train_gen.num_classes} classes: {train_gen.class_indices}")

In [ ]:
# ── STEP 6: Build EfficientNetB0 Transfer Learning Model ───────
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
    drop_connect_rate=0.2
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(train_gen.class_indices), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_acc")]
)
print(f"✅ Model built: {model.count_params():,} total parameters")

In [ ]:
# ── STEP 7: Phase 1 Training (Feature Extraction) ──────────────
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks_p1 = [
    ModelCheckpoint("/content/phase1_best.h5", monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

print("🚀 Starting Phase 1 Training...")
h1 = model.fit(train_gen, epochs=10, validation_data=val_gen, callbacks=callbacks_p1, verbose=1)
print(f"✅ Phase 1 Best Val Accuracy: {max(h1.history['val_accuracy']):.1%}")

In [ ]:
# ── STEP 8: Phase 2 Training (Fine-Tuning Base Layers) ──────────
base_model.trainable = True
# Freeze all except top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_acc")]
)

callbacks_p2 = [
    ModelCheckpoint("/content/cattle_disease_v1.h5", monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-8, verbose=1)
]

print("🚀 Starting Phase 2 Fine-Tuning...")
h2 = model.fit(train_gen, epochs=20, validation_data=val_gen, callbacks=callbacks_p2, initial_epoch=len(h1.history["accuracy"]), verbose=1)
print(f"✅ Phase 2 Best Val Accuracy: {max(h2.history['val_accuracy']):.1%}")

In [ ]:
# ── STEP 9: Test Evaluation & Confusion Matrix ─────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

model.load_weights("/content/cattle_disease_v1.h5")
test_gen.reset()
y_pred = np.argmax(model.predict(test_gen, verbose=1), axis=1)
y_true = test_gen.classes
cls_labels = [CLASS_NAMES[str(i)] for i in range(len(CLASS_NAMES))]

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=cls_labels))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=cls_labels, yticklabels=cls_labels)
plt.title("E-Gowshala Cattle Disease Model — Confusion Matrix", fontweight="bold")
plt.ylabel("True Disease"); plt.xlabel("Predicted Disease")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png", dpi=150)
plt.show()

loss, acc, top2 = model.evaluate(test_gen, verbose=0)
print(f"\n🎯 Final Test Accuracy: {acc:.1%} | Top-2 Accuracy: {top2:.1%}")

In [ ]:
# ── STEP 10: Export TFLite & Package for Download ──────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open("/content/cattle_disease_v1.tflite", "wb") as f:
    f.write(tflite_model)

os.system("zip -j /content/egowshala_model_package.zip /content/cattle_disease_v1.h5 /content/cattle_disease_v1.tflite /content/class_mapping.json /content/confusion_matrix.png")
print("\n🎉 Model package created: /content/egowshala_model_package.zip")

from google.colab import files
files.download("/content/egowshala_model_package.zip")
print("✅ Download triggered automatically!")